### PydanticOutputParser

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
import os

llm = ChatOpenAI(temperature=0, model_name=os.getenv("OPENAI_DEFAULT_MODEL"))

In [2]:
email_conversation = """From: 교육담당자 (edu@sw.or.kr)"
To: 노규남 (bardroh@weable.ai)
Subject: 4월 회원지원 교육 안내

 문의사항이 있으신 경우 담당자에게 연락 바랍니다.

* 선착순 모집이며 조기 마감될 수 있습니다.

* 오프라인과 온라인 수업이 동시에 있는 경우, 동시 접수가 불가능합니다.

* 4월 24일(목) 종합소득세와 소득세 원천징수 온라인 교육이 있습니다.

그외의 교육은 오프라인 교육입니다.

* 온라인으로 교육 참여하시는 경우 교재 파일 링크를 제공해드리며 별도 실물 교재는 제공해 드리지 않습니다.
"""

In [3]:
from itertools import chain
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "다음의 이메일 내용중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)

chain = prompt | llm
aimsg = chain.invoke({"email_conversation": email_conversation})
aimsg.content

'중요한 내용 요약:\n\n- **교육 일정**: 4월 24일(목) 종합소득세와 소득세 원천징수 온라인 교육\n- **모집 방식**: 선착순 모집, 조기 마감 가능\n- **수업 형태**: 오프라인과 온라인 수업 동시 접수 불가\n- **교재 제공**: 온라인 참여 시 교재 파일 링크 제공, 실물 교재는 제공되지 않음\n- **문의**: 문의사항은 담당자에게 연락 필요'

In [4]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 날짜")

parser = PydanticOutputParser(pydantic_object=EmailSummary)
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 날짜", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


In [5]:
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)

prompt = prompt.partial(format=parser.get_format_instructions())

In [6]:
chain = prompt | llm
answer = chain.invoke({"email_conversation": email_conversation,
              "question": "다음의 이메일 내용중 중요한 내용을 추출해 주세요."
})
print(answer)

content='```json\n{\n  "person": "교육담당자",\n  "email": "edu@sw.or.kr",\n  "subject": "4월 회원지원 교육 안내",\n  "summary": "4월 24일(목) 종합소득세와 소득세 원천징수 온라인 교육이 있으며, 선착순 모집으로 조기 마감될 수 있습니다. 오프라인 교육도 진행되며, 온라인 참여 시 실물 교재는 제공되지 않습니다.",\n  "date": "4월 24일"\n}\n```' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 474, 'total_tokens': 589, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_29330a9688', 'id': 'chatcmpl-D0cMsGRHC1hUYeJKA1tptyeR4nz1V', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019be2fe-06ff-7630-9135-cb9cb48c69ce-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 474, 'output_tokens': 115, 'total_tokens': 589, 'input_to

In [7]:
structured_output = parser.parse(answer.content)
print(structured_output)

person='교육담당자' email='edu@sw.or.kr' subject='4월 회원지원 교육 안내' summary='4월 24일(목) 종합소득세와 소득세 원천징수 온라인 교육이 있으며, 선착순 모집으로 조기 마감될 수 있습니다. 오프라인 교육도 진행되며, 온라인 참여 시 실물 교재는 제공되지 않습니다.' date='4월 24일'


In [8]:
chain = prompt | llm | parser
response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)
response

EmailSummary(person='교육담당자', email='edu@sw.or.kr', subject='4월 회원지원 교육 안내', summary='4월 24일(목) 종합소득세와 소득세 원천징수 온라인 교육이 있으며, 선착순 모집으로 조기 마감될 수 있습니다. 오프라인 교육도 진행되며, 온라인 참여 시 교재 파일 링크만 제공됩니다.', date='4월 24일')

In [9]:
llm_with_structered = ChatOpenAI(
    temperature=0, model_name=os.getenv("OPENAI_DEFAULT_MODEL")
).with_structured_output(EmailSummary)

answer = llm_with_structered.invoke(email_conversation)
answer

EmailSummary(person='노규남', email='edu@sw.or.kr', subject='4월 회원지원 교육 안내', summary='4월 24일(목)에 종합소득세와 소득세 원천징수 온라인 교육이 있으며, 선착순 모집으로 조기 마감될 수 있습니다. 오프라인 교육도 진행되며, 온라인 참여 시 교재 파일 링크가 제공됩니다. 동시 접수는 불가능합니다.', date='2023-04-24')

### CommaSeparatedListOutputParser

In [10]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()
format_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [11]:

prompt = PromptTemplate(
    template="List 10 {subject}.\n{format_instructions}",
    input_variables=["subject"], 
    partial_variables={"format_instructions": format_instructions}, # 미리 값이 지정되어 있으나 오버라이드 가능한 변수
)

model = ChatOpenAI(temperature=0, model_name=os.getenv("OPENAI_DEFAULT_MODEL"))

chain = prompt | model | output_parser

In [12]:
chain.invoke({"subject": "미국 국립공원"})

['옐로스톤',
 '그랜드 캐니언',
 '요세미티',
 '자이언',
 '스모키 마운틴',
 '아카디아',
 '시퀼라 국립공원',
 '올림픽',
 '글레이셔',
 '세쿼이아']

### with_structed_output

In [13]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

class QAResponse(BaseModel):
    """질문-답변 응답"""
    answer: str = Field(description="사용자의 질문에 대한 답변")
    source: str = Field(description="출처 URL")

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 도움이 되는 어시스턴트입니다. 답변과 출처를 제공하세요."),
    ("user", "{question}")
])

model = ChatOpenAI(model="gpt-4o-mini")
structured_model = model.with_structured_output(QAResponse)

chain = prompt | structured_model

response = chain.invoke({"question": "OpenAI의 창업자는 누구인가요?"})

print(f"답변: {response.answer}")
print(f"출처: {response.source}")

답변: OpenAI의 창립자는 엘론 머스크, 샘 알트만, 그렉 브록만, 일야 수츠케버, 워지치 자레프스키, 그리고 존 매카시 등의 인물들이 포함되어 있습니다. 이들은 인공지능 기술의 발전과 안전성을 목표로 OpenAI를 설립하였습니다.
출처: https://openai.com/about/


### JsonOutputParser

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

In [15]:
model = ChatOpenAI(temperature=0, model_name=os.getenv("OPENAI_DEFAULT_MODEL"))

In [16]:
class Topic(BaseModel):
    description: str = Field(description="주제에 대한 간결한 설명")
    keywords: str = Field(description="설명에 대한 주요 키워드(2개 이상)")

In [17]:
question = "도널드 트럼프의 외교정책에 대해서 설명해주세요."

parser = JsonOutputParser(pydantic_object=Topic)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간단하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser  

chain.invoke({"question": question})

{'description': "도널드 트럼프의 외교정책은 '미국 우선주의'를 중심으로 하며, 무역 협정 재협상, NATO 동맹국에 대한 방위비 분담 요구, 이란 핵 합의 탈퇴, 중국과의 무역 전쟁 등이 포함된다.",
 'keywords': '미국 우선주의, 무역 전쟁'}

In [18]:
question = "도널드 트럼프의 외교정책에 대해서 설명해주세요."
parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간단하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser

response = chain.invoke({"question": question})

print(response)

{'도널드_트럼프의_외교정책': {'주요_특징': ['미국 우선주의: 미국의 이익을 최우선으로 고려', '무역전쟁: 중국과의 무역 갈등을 통해 관세 인상', '북한과의 대화: 김정은과의 정상회담을 통한 비핵화 논의', 'NATO와의 관계: NATO 동맹국들에게 방위비 분담 증액 요구', '중동 정책: 이스라엘과의 관계 강화 및 이란 핵 협정 탈퇴'], '목표': ['미국의 경제 성장 촉진', '국가 안보 강화', '국제 관계에서의 미국의 영향력 증대'], '비판': ['다자간 협력 약화', '동맹국과의 관계 긴장', '인권 문제에 대한 무관심']}}


In [19]:
question = "도널드 트럼프의 외교정책에 대해서 설명해주세요. 설명은 'description'에, 키워드는 'keywords'에 기록해주세요."

parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간단하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser

response = chain.invoke({"question": question})

print(response)

{'description': "도널드 트럼프의 외교정책은 '미국 우선주의'를 중심으로 하며, 무역 협정 재협상, NATO 및 동맹국에 대한 부담 분담 요구, 이란 핵 합의 탈퇴, 북한과의 정상 회담 등으로 특징지어집니다. 또한, 중국과의 무역 전쟁을 통해 무역 불균형 해소를 목표로 했습니다.", 'keywords': ['미국 우선주의', '무역 협정', 'NATO', '이란 핵 합의', '북한 정상 회담', '중국 무역 전쟁']}


### EnumOutputParser

In [20]:
from enum import Enum
from pydantic import BaseModel
from langchain_openai import ChatOpenAI

class Colors(str, Enum):
    RED = "빨간색"
    GREEN = "초록색"
    BLUE = "파란색"

class ColorResponse(BaseModel):
    color: Colors

model = ChatOpenAI(model="gpt-4o-mini")
structured_model = model.with_structured_output(ColorResponse)

result = structured_model.invoke("선호하는 색상은?")
print(result)  # ColorResponse(color=<Colors.RED: '빨간색'>)
print(result.color.value)  # '빨간색'

color=<Colors.BLUE: '파란색'>
파란색


In [21]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

prompt = PromptTemplate.from_template(
    """다음의 물체는 어떤 색깔인가요?

Object: {object}

Instructions: {instructions}"""
).partial(instructions=parser.get_format_instructions())

chain = prompt | ChatOpenAI() | parser

In [22]:
response = chain.invoke({"object": "하늘"}) 
print(response)

{'object': '하늘', 'color': '파란색'}


### Custom Output Parser

In [23]:
from langchain_core.output_parsers import BaseOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

In [24]:
class CommaSeparatedListOutputParser(BaseOutputParser):
    """
    쉼표로 구분된 항목 목록을 파싱하는 간단한 출력 파서
    """
    def parse(self, text: str) -> list:
        cleaned_text = text.strip()
        if not cleaned_text:
            return []
        items = [item.strip() for item in cleaned_text.split(",")]
        return items

In [25]:
template = """
다음 주제에 관련된 항목을 5개 나열해주세요: {topic}

항목은 쉼표로 구분해 주세요.
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["topic"],
)

output_parser = CommaSeparatedListOutputParser()

In [26]:
import os

llm = ChatOpenAI(temperature=0, model_name=os.getenv("OPENAI_DEFAULT_MODEL"))
chain = prompt | llm | output_parser

In [27]:
result = chain.invoke({"topic": "인공지능의 응용 분야"})
print(result) 
print(type(result)) 

['자연어 처리', '이미지 인식', '자율주행차', '의료 진단', '추천 시스템']
<class 'list'>


In [28]:
result2 = chain.invoke({"topic": "한국의 대표 음식"})
print(result2)

['김치', '비빔밥', '불고기', '떡볶이', '삼겹살']
